[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.4_speculative_decoding/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/05_optimization/04.4_speculative_decoding/lab.ipynb)

# 4.4 Lab: Speculative Decoding

Simulate the draft-verify loop, compute acceptance rate impact on speedup, and visualize when speculative decoding helps vs hurts.

In [ ]:
# Install dependencies via subprocess (avoids kernel restart)
import subprocess
# numpy: numerical computation for simulation
# matplotlib: plotting speedup curves and comparisons
subprocess.run(['pip', 'install', '-q', 'numpy', 'matplotlib'], check=True)

# Import numerical library for random sampling and math
import numpy as np
# Import plotting library for visualization
import matplotlib.pyplot as plt

# Fix random seed for reproducible simulation results
np.random.seed(42)
# Use clean grid-based plot style
plt.style.use('seaborn-v0_8-whitegrid')
# Confirm environment is ready
print('Setup complete')

## 1. Speedup vs Acceptance Rate

Core formula: expected tokens per round = (1 - α^(γ+1)) / (1 - α), where α is acceptance rate and γ is draft count.

In [ ]:
def expected_tokens_per_round(alpha, gamma):
    """Compute expected accepted tokens per speculative decoding round."""
    # alpha: per-token acceptance probability (0 to 1)
    # gamma: number of draft tokens proposed each round
    if alpha == 1.0:
        # Perfect acceptance: all draft tokens pass
        return gamma + 1
    # Geometric series formula for expected accepted tokens
    # Derives from: P(accept k) = alpha^k * (1-alpha) for k < gamma
    return (1 - alpha**(gamma + 1)) / (1 - alpha)


def speculative_speedup(alpha, gamma, draft_cost_fraction=0.1):
    """Compute wall-clock speedup including draft model overhead."""
    # alpha: acceptance rate per token
    # gamma: speculative tokens per round
    # draft_cost_fraction: draft time as fraction of target pass
    #
    # Expected tokens produced in this round
    tokens = expected_tokens_per_round(alpha, gamma)
    # Cost: 1 target verification + gamma cheap draft steps
    cost = 1 + gamma * draft_cost_fraction
    # Speedup = tokens gained per unit cost
    return tokens / cost


# Create 100 evenly spaced acceptance rate values from 0 to 0.99
alphas = np.linspace(0.0, 0.99, 100)

# Test four different draft lengths (gamma)
gamma_values = [2, 4, 6, 8]
# Compute speedup curve for each gamma across all acceptance rates
speedups = {g: [speculative_speedup(a, g) for a in alphas] for g in gamma_values}

# Create figure for speedup visualization
fig, ax = plt.subplots(figsize=(8, 5))
# Plot one curve per gamma value
for g in gamma_values:
    # Each line shows speedup as function of acceptance rate
    ax.plot(alphas, speedups[g], label=f'γ={g}', linewidth=2)

# Draw breakeven line: speedup=1 means no benefit
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Breakeven')
# Shade region below breakeven (speculation hurts)
ax.fill_between(alphas, 0, 1, alpha=0.05, color='red')

# Label axes and title
ax.set_xlabel('Acceptance Rate (α)')
ax.set_ylabel('Speedup vs Standard Decoding')
ax.set_title('Speculative Decoding Speedup by Draft Count (γ)')
# Add legend to identify curves
ax.legend()
# Cap y-axis for readability
ax.set_ylim(0, 8)
plt.tight_layout()
# Save figure to disk
plt.savefig('speedup_vs_acceptance.png', dpi=150)
plt.show()
# Print specific data points for reference
print(f'At α=0.8, γ=4: {speculative_speedup(0.8, 4):.2f}x speedup')
print(f'At α=0.5, γ=4: {speculative_speedup(0.5, 4):.2f}x speedup')

## 2. Draft-Verify Loop Simulation

Simulate the sequential process: draft proposes tokens, target verifies, accepted tokens go to output.

In [ ]:
def simulate_speculative_decoding(total_tokens, gamma, alpha, target_time_ms, draft_fraction):
    """Simulate speculative decoding and return timing breakdown."""
    # total_tokens: how many output tokens to generate
    # gamma: draft tokens proposed per round
    # alpha: probability each draft token is accepted
    # target_time_ms: milliseconds for one target forward pass
    # draft_fraction: draft cost as fraction of target cost
    #
    # Initialize counters
    tokens_generated = 0  # Running count of output tokens
    rounds = 0  # Number of draft-verify rounds executed
    total_draft_time = 0.0  # Accumulated draft model time
    total_verify_time = 0.0  # Accumulated target verification time
    accepted_counts = []  # Track per-round acceptance for analysis
    
    # Main generation loop: keep going until we have enough tokens
    while tokens_generated < total_tokens:
        rounds += 1
        # Draft phase: small model generates gamma candidate tokens
        # Cost = gamma * (fraction of target time)
        draft_time = gamma * target_time_ms * draft_fraction
        total_draft_time += draft_time
        
        # Verify phase: target model processes all candidates in parallel
        # This costs exactly one target forward pass regardless of gamma
        total_verify_time += target_time_ms
        
        # Acceptance: check each draft token sequentially
        # First rejection stops the acceptance chain
        accepted = 0
        for i in range(gamma):
            # Random draw simulates target probability check
            if np.random.random() < alpha:
                # Token accepted: target agrees with draft
                accepted += 1
            else:
                # Token rejected: stop accepting further tokens
                break
        # Guaranteed minimum: 1 token from corrected distribution
        tokens_this_round = accepted + 1
        # Add produced tokens to running total
        tokens_generated += tokens_this_round
        # Record for statistics
        accepted_counts.append(accepted)
    
    # Baseline comparison: standard autoregressive decoding
    # Each token costs one full target forward pass
    baseline_time = total_tokens * target_time_ms
    # Speculative total = all draft time + all verify time
    speculative_time = total_draft_time + total_verify_time
    
    # Return comprehensive results dict
    return {
        'rounds': rounds,
        'baseline_ms': baseline_time,
        'speculative_ms': speculative_time,
        'speedup': baseline_time / speculative_time,
        'avg_accepted': np.mean(accepted_counts),
    }


# Configuration: Llama-70B target + Llama-8B draft
TARGET_TIME_MS = 30  # 30ms per 70B forward pass on A100
DRAFT_FRACTION = 0.1  # 8B draft costs ~10% of 70B target
TOTAL_TOKENS = 200  # Simulate generating 200 output tokens

# Test across a range of acceptance rates
sim_alphas = [0.4, 0.6, 0.7, 0.8, 0.9, 0.95]
# Collect simulation results
sim_results = []
for a in sim_alphas:
    # Run 50 trials per acceptance rate for stable estimates
    trial_speedups = []
    for _ in range(50):
        # Run one full simulation
        result = simulate_speculative_decoding(
            TOTAL_TOKENS, gamma=4, alpha=a,
            target_time_ms=TARGET_TIME_MS, draft_fraction=DRAFT_FRACTION
        )
        # Record speedup from this trial
        trial_speedups.append(result['speedup'])
    # Store mean and std across trials
    sim_results.append({
        'alpha': a,
        'mean_speedup': np.mean(trial_speedups),
        'std_speedup': np.std(trial_speedups),
    })

# Print results table
print(f'{"α":>6} | {"Speedup":>8} | {"Std":>6}')
print('-' * 28)
for r in sim_results:
    # Display acceptance rate, mean speedup, and variance
    print(f'{r["alpha"]:>6.2f} | {r["mean_speedup"]:>8.2f}x | {r["std_speedup"]:>6.3f}')

## 3. Speculation vs Batch Size

Both speculation and batching amortize weight reads. This shows diminishing returns as batch grows.

In [ ]:
def effective_latency_per_token(batch_size, use_speculative, alpha=0.8, gamma=4,
                                 model_size_gb=16, bandwidth_gbps=2000):
    """Estimate per-token latency with and without speculation."""
    # batch_size: concurrent sequences sharing one weight read
    # model_size_gb: target model weights in GB
    # bandwidth_gbps: GPU HBM bandwidth (A100=2TB/s, H100=3.35TB/s)
    #
    # Time to read all model weights from HBM once
    weight_read_ms = (model_size_gb / bandwidth_gbps) * 1000
    # Standard decode: one read serves all batch_size sequences
    standard_per_token = weight_read_ms / batch_size
    
    if not use_speculative:
        # No speculation: just batch amortization
        return standard_per_token
    
    # With speculation: more tokens per verification round
    tokens_per_round = expected_tokens_per_round(alpha, gamma)
    # Draft overhead: gamma small-model reads
    draft_overhead_ms = gamma * 0.1 * weight_read_ms
    # Cost per token = (verify + draft) / (tokens_per_round * batch)
    speculative_per_token = (weight_read_ms + draft_overhead_ms) / (tokens_per_round * batch_size)
    return speculative_per_token


# Test batch sizes from 1 (interactive) to 64 (throughput)
batch_sizes = [1, 2, 4, 8, 16, 32, 64]

# Compute latency for both modes across all batch sizes
latency_standard = [effective_latency_per_token(b, False) for b in batch_sizes]
latency_speculative = [effective_latency_per_token(b, True) for b in batch_sizes]

# Calculate percentage benefit of speculation at each batch size
benefit_pct = [(s - sp) / s * 100 for s, sp in zip(latency_standard, latency_speculative)]

# Create side-by-side comparison plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: absolute latency comparison
ax1.plot(batch_sizes, latency_standard, 'o-', label='Standard', linewidth=2)
ax1.plot(batch_sizes, latency_speculative, 's-', label='Speculative (α=0.8)', linewidth=2)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Per-Token Latency (ms)')
ax1.set_title('Latency: Standard vs Speculative')
# Log scale for batch axis (powers of 2)
ax1.set_xscale('log', base=2)
ax1.legend()

# Right panel: benefit percentage shrinks with batch
ax2.bar(range(len(batch_sizes)), benefit_pct, color='#dbeafe', edgecolor='#000')
# Label x-axis with actual batch sizes
ax2.set_xticks(range(len(batch_sizes)))
ax2.set_xticklabels(batch_sizes)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Speculation Benefit (%)')
ax2.set_title('Diminishing Returns with Larger Batches')
# Threshold line: below 10% benefit, speculation not worth the complexity
ax2.axhline(y=10, color='red', linestyle='--', alpha=0.5, label='<10% threshold')
ax2.legend()

plt.tight_layout()
# Save comparison figure
plt.savefig('batch_vs_speculation.png', dpi=150)
plt.show()
# Key finding printed for quick reference
print('Key insight: speculation adds <10% benefit above batch=16')

## Key Takeaways

1. Speedup scales with acceptance rate: α=0.8, γ=4 gives ~2.4x
2. Draft model cost matters: keep draft < 10% of target time
3. Speculation and batching are substitutes: at batch > 8, speculation adds minimal value
4. Best use case: interactive, low-batch, predictable-output workloads